# CNN–ConvLSTM: replication of Mahdiyasa et al. (2026)

## 1. Purpose and replication specification

Evaluate the saved CNN–ConvLSTM using **Jan 2020–Sep 2023 training → Oct–Dec 2023 validation → 2024 evaluation**. The paper uses all 2020–2023 for fitting; this run reserves the final quarter for validation and does not perform a full-year refit. This is a documented deviation from the paper-aligned training split. This is a research replication with documented assumptions. Both 2024 and 2025 outcomes have already been inspected. Compare with section 4 of `base_models.ipynb`; spatial sequences provide information absent from those tabular inputs.

Read [the replication specification](docs/conv-lstm-replication.md) before training. Page references count the PDF cover as page 1.

| Setting | Paper evidence | Choice here |
| --- | --- | --- |
| Grid and encoding | §2.2.2, pp. 14–16 | Existing 5 km candidate BIG grid, 4 continuous + 17 one-hot predictors |
| Sequence and target | §2.3.1, pp. 19–20 gives consecutive maps but no length or offset | **Assumption:** seven days through target day; same-day detection, horizon 0, no historical-fire predictor |
| Architecture | §2.3.2, p. 22 | ConvLSTM 32 → Conv2D 64 → Conv2D 32 → sigmoid 1; exact kernels/normalization/dropout below |
| Loss | Eq. (2), p. 17; Table 2, p. 23 | Positive weight 75; masked mean over valid cell-days |
| Training | §2.3.2, p. 22 | Batch 2, up to 70 epochs, patience 20; Adam 0.001 is an assumption |
| Epoch selection | Validation membership/monitor unspecified | Fit Jan 2020–Sep 2023; select the lowest Oct–Dec 2023 weighted validation loss; evaluate that saved model on 2024 |
| Calibration | §2.3.1, p. 21; cohort unspecified | Raw results first; optional Jan–Jun 2024 isotonic fitting → Jul–Dec evaluation |

The published weight 75 was selected using the testing period. Retaining it does not establish independent validation of that choice. We do not rerun the weight search. The paper's three- and ten-day forecasts cannot be recovered from the described timing/availability details; this notebook makes no forecast claim.

### 1.1 Runtime and fixed reconstruction choices

Install the project requirements and optional `requirements-conv-lstm.txt` in the notebook environment. The training stage is complete for the saved run: epoch 23 was selected, and early stopping ended the run at epoch 43. The notebook defaults to `RUN_MODE = "saved"`: it reuses the completed run, displays its combined history, and loads the best model for evaluation. It does not train again. Choose `RUN_MODE = "new"` only to prepare a separate run and fit a fresh model. The separate isotonic calibration experiment is enabled by default. Raw data, arrays and weights are local gitignored artifacts; a fresh clone needs the documented data preparation or its own training run.

In [1]:
import gc
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display
from peatguard.study import find_root, load_study
from peatguard.datasets import CONTINUOUS, PREDICTORS, _grid
from peatguard.inventory import sha256
from peatguard.conv_lstm import (
    SpatialStore, prepare_store, build_model, batch_dataset, predict_cells, evaluate_scores,
)

ROOT = find_root()
DATA_DIR = ROOT / "data/interim/model_table"
LENGTH, BATCH_SIZE, POSITIVE_WEIGHT = 7, 2, 75.0
MAX_EPOCHS, PATIENCE, SEED, LEARNING_RATE = 70, 20, 42, 0.001
# "saved" reuses the completed model; "new" deliberately starts fresh training.
RUN_MODE = "saved"
RUN_CALIBRATION = True
SAVED_RUN = ROOT / "data/interim/conv_lstm/20261027T155222671757Z"
if RUN_MODE == "saved":
    RUN_DIR = SAVED_RUN
    for name in ["run.json", "development_result.json", "combined_selection_history.csv",
                 "development_best.weights.h5", "development_scaling/scaling.json"]:
        if not (RUN_DIR / name).exists():
            raise FileNotFoundError(f"Saved run artifact missing: {RUN_DIR / name}. "
                                    "Supply the saved run or choose RUN_MODE = 'new'.")
elif RUN_MODE == "new":
    RUN_DIR = ROOT / "data/interim/conv_lstm" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    RUN_DIR.mkdir(parents=True, exist_ok=False)
else:
    raise ValueError("RUN_MODE must be 'saved' or 'new'.")
tf.keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__, "Keras:", tf.keras.__version__)
print("Run artifacts:", RUN_DIR.relative_to(ROOT))

TensorFlow: 2.21.0 Keras: 3.15.1
Run artifacts: data/interim/conv_lstm/20261027T155222671757Z


## 2. Preparation: current export, spatial contract, training-only scaling

Use only the current `train_2020_2024.parquet`. Verify its checksum, build identity and saved scaling; do not load 2025 or silently fall back to an older export. `_grid` checks the current grid configuration, stable cell centers and boundary hashes. The north-up map uses `row = max(grid_row) − grid_row`, `column = grid_col − min(grid_col)`.

The saved `_encode` contract is an **unclipped affine** min–max transform. Invert it using the source scaler, fit new ranges using Jan 2020–Sep 2023 only, and apply those ranges without clipping. Validation (Oct–Dec 2023) and evaluation (2024) use that same September-fitted scaler. The older `full_training_scaling` arrays are retained as historical artifacts and are not inputs to this saved-model evaluation. A training-constant feature stays zero, matching dataset preparation. Current source mappings originate in 2020; this reconstruction does not rebuild spatial sampling.

Compact daily features are stored on disk, not expanded into every overlapping sequence. A separate domain mask and daily-observed mask are input channels (a documented addition to the paper). Missing rows remain unknown; target supervision also requires complete feature history at that cell.

In saved mode, open the existing September-fitted arrays read-only. In new mode, reconstruct scaling from the verified export and create new arrays. Both modes use the same predictor order, calendar and spatial addresses. No previous dataset or model file is overwritten.

In [2]:
export_info = json.loads((DATA_DIR / "prepared_dataset.json").read_text())
source_build = ROOT / export_info["build_directory"]
source_manifest = json.loads((source_build / "manifest.json").read_text())
source_scaler = json.loads((source_build / "scaler.json").read_text())
table_path = DATA_DIR / "train_2020_2024.parquet"
assert export_info["build_id"] == source_manifest["build_id"]
assert export_info["partitions"]["train"]["file"] == table_path.name
assert export_info["partitions"]["train"]["start"] == "2020-01-01"
assert export_info["partitions"]["train"]["end"] == "2024-12-31"
assert sha256(table_path) == export_info["partitions"]["train"]["sha256"]
assert source_manifest["predictors"] == PREDICTORS
assert source_scaler == source_manifest["fitted_ranges"]
assert sha256(source_build / "scaler.json") == source_manifest["files"]["scaler.json"]

study = load_study(root=ROOT, profile="legacy", as_of="2026-10-25")
cells, _, _, _ = _grid(study)
for relative in ["data/interim/study_grid_5km/study_grid_5km.csv",
                 "data/interim/study_grid_5km/manifest.json"]:
    assert sha256(ROOT / relative) == source_manifest["descriptor"]["sources"][relative]
if RUN_MODE == "saved":
    scaling_dir = RUN_DIR / "development_scaling"
    scaling_record = json.loads((scaling_dir / "scaling.json").read_text())
    assert scaling_record["fit_start"] == "2020-01-01"
    assert scaling_record["fit_end"] == "2023-09-30"
    assert scaling_record["predictors"] == PREDICTORS
    store = SpatialStore(
        pd.date_range("2020-01-01", "2024-12-31"),
        cells.sort_values("cell_id").reset_index(drop=True),
        np.load(scaling_dir / "features.npy", mmap_mode="r"),
        np.load(scaling_dir / "observed.npy", mmap_mode="r"),
        np.load(scaling_dir / "labels.npy", mmap_mode="r"),
        scaling_record["fitted_ranges"],
    )
else:
    store = prepare_store(table_path, cells, source_scaler, RUN_DIR / "development_scaling",
                          fit_end="2023-09-30")
display(pd.DataFrame(store.scaling).T)
print(f"{len(store.dates):,} days; {len(store.cells):,} study cells; maps {store.height} × {store.width}")
print(f"Known feature cell-days: {int(store.observed.sum()):,}")

if RUN_MODE == "saved":
    run_spec = json.loads((RUN_DIR / "run.json").read_text())
    assert run_spec["source_build"] == export_info
else:
    run_spec = {
        "source_build": export_info, "paper_sha256": sha256(ROOT / "papers/mahdiyasa2026.pdf"),
        "sequence_length": LENGTH, "horizon_days": 0, "historical_fire_input": False,
        "mask_channels": ["domain", "observed"], "positive_weight": POSITIVE_WEIGHT,
        "batch_size": BATCH_SIZE, "max_epochs": MAX_EPOCHS, "patience": PATIENCE,
        "seed": SEED, "optimizer": "Adam", "learning_rate": LEARNING_RATE,
        "tensorflow": tf.__version__, "keras": tf.keras.__version__,
        "specification": "docs/conv-lstm-replication.md",
        "specification_sha256": sha256(ROOT / "docs/conv-lstm-replication.md"),
        "implementation_sha256": sha256(ROOT / "peatguard/conv_lstm.py"),
        "fit_interval": ["2020-01-01", "2023-09-30"],
        "validation_interval": ["2023-10-01", "2023-12-31"],
        "evaluation_interval": ["2024-01-01", "2024-12-31"],
        "full_year_refit": False,
        "status": "prepared; not trained",
    }
    (RUN_DIR / "run.json").write_text(json.dumps(run_spec, indent=2) + "\n")


,min,max,constant
precipitation_mm_day,0.0,127.191032,False
temperature_2m_C,21.640076,31.240784,False
soil_water_layer1_fraction,0.052579,0.604857,False
distance_to_built_m,3.723917,30903.356802,False


1,827 days; 3,512 study cells; maps 168 × 217
Known feature cell-days: 6,416,424


## 3. Sequences: verify one batch before training

A target on day *t* uses feature maps *t−6,…,t*, with no fire-label channel. The first six calendar days have insufficient history and are excluded. History crosses calendar-year boundaries; January 2024 may use December 2023 features. Only target dates determine split membership. Missing days are retained as empty calendar positions and never bridged by compressing time.

A valid target cell has a known label and all seven local feature observations. Missing neighboring features are zero-filled with observed mask 0; outside-domain cells have domain mask 0. These cells contribute neither positive nor negative training examples. Convolutions and batch normalization still see the padded background, a stated reconstruction limitation.

In [3]:
train_targets = store.targets("2020-01-01", "2023-09-30", LENGTH)
validation_targets = store.targets("2023-10-01", "2023-12-31", LENGTH)
test_targets = store.targets("2024-01-01", "2024-12-31", LENGTH)

x_batch, y_batch = store.batch(train_targets[:BATCH_SIZE], LENGTH)

for b, target in enumerate(train_targets[:BATCH_SIZE]):
    sequence_dates = store.dates[target-LENGTH+1:target+1]

    for step, day in enumerate(range(target-LENGTH+1, target+1)):
        np.testing.assert_allclose(x_batch[b, step, store.rows, store.cols, :21], store.features[day])
        np.testing.assert_array_equal(x_batch[b, step, store.rows, store.cols, -1], store.observed[day])
    np.testing.assert_array_equal(y_batch[b, store.rows, store.cols, 0], np.maximum(store.labels[target], 0))
    np.testing.assert_array_equal(y_batch[b, store.rows, store.cols, 1], store.valid_cells(target, LENGTH))
print("Input:", x_batch.shape, "Packed label/mask:", y_batch.shape)
print("First sequence:", store.dates[train_targets[0]-LENGTH+1], "through", store.dates[train_targets[0]])

coverage = []
for role, targets, start, end in [
    ("Training", train_targets, "2020-01-01", "2023-09-30"),
    ("Validation", validation_targets, "2023-10-01", "2023-12-31"),
    ("Evaluation", test_targets, "2024-01-01", "2024-12-31"),
]:
    requested = len(pd.date_range(start, end)) * len(store.cells)
    valid = sum(int(store.valid_cells(i, LENGTH).sum()) for i in targets)
    positives = sum(int((store.labels[i][store.valid_cells(i, LENGTH)] == 1).sum()) for i in targets)
    coverage.append({"Role": role, "Requested cell-days": requested, "Scored cell-days": valid,
                     "Excluded cell-days": requested-valid, "Positive cell-days": positives})
display(pd.DataFrame(coverage))

Input: (2, 7, 168, 217, 23) Packed label/mask: (2, 168, 217, 2)
First sequence: 2020-01-01 00:00:00 through 2020-01-07 00:00:00


,Role,Requested cell-days,Scored cell-days,Excluded cell-days,Positive cell-days
0,Training,4807928,4786856,21072,607
1,Validation,323104,323104,0,332
2,Evaluation,1285392,1285392,0,70


## 4. Architecture and masked weighted loss

The explicit implementation in §2.3.2 (PDF p. 22) starts with ConvLSTM, despite the conceptual discussion describing spatial extraction before recurrence. Follow that explicit order:

1. ConvLSTM2D: 32 filters, 3×3, tanh, sigmoid recurrence → batch normalization → dropout 0.30.
2. Conv2D: 64 filters, 3×3, ReLU → batch normalization → dropout 0.30.
3. Conv2D: 32 filters, 3×3, ReLU → batch normalization.
4. Conv2D: 1 filter, 1×1, sigmoid.

Same padding and last-state output preserve the map dimensions; no pooling or additional CNN is invented. The loss is `sum(mask × weighted_BCE) / sum(mask)`: positives get weight **75**, negatives **1**. The weight is a loss penalty, not 75% probability. An entirely masked batch has zero loss.

### 4.1 Smoke check

Check full-map inference and one gradient update on a small spatial crop. This disposable model is discarded before scientific training; its predictions are not evaluation results.

In [4]:
if RUN_MODE == "new":
    tf.keras.utils.set_random_seed(SEED)
    smoke_model = build_model(LENGTH, 23, POSITIVE_WEIGHT, LEARNING_RATE,
                              spatial_shape=(store.height, store.width))
    smoke_model.summary()
    full_prediction = smoke_model(x_batch[:1], training=False).numpy()
    
    # Crop around an observed study cell to keep the gradient smoke test inexpensive.
    r, c = int(store.rows[0]), int(store.cols[0])
    r0, c0 = max(0, r-8), max(0, c-8)
    x_crop = x_batch[:, :, r0:r0+16, c0:c0+16]
    y_crop = y_batch[:, r0:r0+16, c0:c0+16]
    
    crop_model = build_model(LENGTH, 23, POSITIVE_WEIGHT, LEARNING_RATE,
                             spatial_shape=x_crop.shape[2:4])
    crop_model.set_weights(smoke_model.get_weights())
    smoke_loss = float(crop_model.train_on_batch(x_crop, y_crop))
    
    print("Full-map inference passed; crop training loss:", smoke_loss)
    del smoke_model, crop_model, full_prediction, x_batch, y_batch, x_crop, y_crop
    tf.keras.backend.clear_session()
    gc.collect()
    run_spec["status"] = "preparation and smoke checks passed; scientific training pending"
    (RUN_DIR / "run.json").write_text(json.dumps(run_spec, indent=2) + "\n")
else:
    print("Saved mode: disposable training smoke check skipped; completed results are retained.")

Saved mode: disposable training smoke check skipped; completed results are retained.


## 5. Training: fit through September 2023, validate on October–December 2023

The saved model learned from **Jan 2020–Sep 2023**. **Oct–Dec 2023 was used for validation**, including early-stopping and checkpoint selection. Scaling was fitted only through September. This validation arrangement is a reconstruction choice because the paper does not identify its validation cohort. It leaves less data for weight fitting than the paper's full 2020–2023 fit.

The completed run stopped at epoch 43 after 20 epochs without improvement; the lowest validation weighted loss was at epoch 23 (CSV index 22). We retain those restored best weights and evaluate them on 2024. **There is no additional full-year refit in this experiment.** Weighted training/validation loss is not calibrated probability error.

### 5.1 Train a new model or display the saved training history

There is one training cell. In **saved mode**, it reads the completed history and summary without calling `fit`. In **new mode**, it trains using the prepared data, saves best/latest checkpoints each epoch, and writes the selected epoch and result summary.

The recorded run was manually interrupted during epoch 2 and resumed from partial-epoch weights. Some batches were repeated and early-stopping patience restarted on resume. The combined CSV preserves its complete epoch-level history. The interrupted notebook and temporary resume steps are archived locally with the run; the traceback is not part of this finished notebook. A fresh continuous run need not reproduce its scores exactly.

The complete recorded epoch history and small evaluation tables are preserved in the tracked [results folder](results/conv_lstm/validation_2023q4_evaluation_2024/README.md). The notebook keeps a compact training summary rather than the full epoch console log. The CSV is zero-based: index 22 corresponds to displayed epoch 23.

In [5]:
if RUN_MODE == "new":
    development_store = store
    fit_targets = development_store.targets("2020-01-01", "2023-09-30", LENGTH)
    validation_targets = development_store.targets("2023-10-01", "2023-12-31", LENGTH)
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    development_model = build_model(
        LENGTH, len(PREDICTORS)+2, POSITIVE_WEIGHT, LEARNING_RATE,
        spatial_shape=(store.height, store.width),
    )
    selection_history = development_model.fit(
        batch_dataset(development_store, fit_targets, LENGTH, BATCH_SIZE),
        validation_data=batch_dataset(development_store, validation_targets, LENGTH, BATCH_SIZE),
        epochs=MAX_EPOCHS,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss", mode="min", patience=PATIENCE, restore_best_weights=True,
            ),
            tf.keras.callbacks.ModelCheckpoint(
                str(RUN_DIR / "development_best.weights.h5"), monitor="val_loss", mode="min",
                save_best_only=True, save_weights_only=True,
            ),
            tf.keras.callbacks.ModelCheckpoint(
                str(RUN_DIR / "latest_development.weights.h5"), save_weights_only=True,
            ),
            tf.keras.callbacks.TerminateOnNaN(),
            tf.keras.callbacks.CSVLogger(str(RUN_DIR / "selection_history.csv")),
        ],
        shuffle=False, verbose=2,
    )
    combined_history = pd.read_csv(RUN_DIR / "selection_history.csv")
    assert len(combined_history) > 0
    assert np.isfinite(combined_history[["loss", "val_loss"]]).all().all()
    combined_history.to_csv(RUN_DIR / "combined_selection_history.csv", index=False)
    best_row = combined_history.loc[combined_history["val_loss"].idxmin()]
    selected_epochs = int(best_row["epoch"]) + 1
    development_result = {
        "stage": "validation-selected development model",
        "status": "training complete; raw 2024 evaluation pending",
        "last_completed_epoch": int(combined_history["epoch"].max()) + 1,
        "selected_epochs": selected_epochs,
        "best_validation_weighted_loss": float(best_row["val_loss"]),
        "training_weighted_loss_at_best_epoch": float(best_row["loss"]),
        "best_weights_file": "development_best.weights.h5",
        "continuation_checkpoint": "latest_development.weights.h5",
        "history_file": "combined_selection_history.csv",
        "full_year_refit": False, "evaluation_2024_complete": False,
    }
    (RUN_DIR / "development_architecture.json").write_text(development_model.to_json() + "\n")
    (RUN_DIR / "development_result.json").write_text(json.dumps(development_result, indent=2) + "\n")
    run_spec.update(status="training complete; raw 2024 evaluation pending",
                    selected_epochs=selected_epochs, development_result="development_result.json")
    (RUN_DIR / "run.json").write_text(json.dumps(run_spec, indent=2) + "\n")
else:
    combined_history = pd.read_csv(RUN_DIR / "combined_selection_history.csv")
    development_result = json.loads((RUN_DIR / "development_result.json").read_text())
    best_row = combined_history.loc[combined_history["val_loss"].idxmin()]
    selected_epochs = int(best_row["epoch"]) + 1
    assert selected_epochs == development_result["selected_epochs"]
    print("Saved mode: no training performed.")

print(f"Completed epochs: {int(combined_history['epoch'].max()) + 1}; selected epoch: {selected_epochs}")
display(pd.DataFrame([{
    "Completed epochs": int(combined_history["epoch"].max()) + 1,
    "Selected epoch": selected_epochs,
    "Training loss at selected epoch": float(best_row["loss"]),
    "Validation loss at selected epoch": float(best_row["val_loss"]),
}]))


Saved mode: no training performed.
Completed epochs: 43; selected epoch: 23


,Completed epochs,Selected epoch,Training loss at selected epoch,Validation loss at selected epoch
0,43,23,0.01914,0.153256


### 5.2 Load the saved validation-selected model — no training

Load `development_best.weights.h5`, which was saved after early stopping restored the best weights (epoch 23). `latest_development.weights.h5` is the stopping-epoch checkpoint (epoch 43), so it is not the selected model.

Reload the **September-fitted** feature arrays from `development_scaling`, verify the original export/grid identity, and freeze the network for inference. This cell can be run after reopening the notebook without rerunning preparation or any training cell. Artifacts are saved under `development_2024_evaluation/` in the original run folder. The optimizer slots in the best-weight file are from the stopping point and are irrelevant to inference.

In [6]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display
from peatguard.study import find_root, load_study
from peatguard.datasets import PREDICTORS, _grid
from peatguard.inventory import sha256
from peatguard.conv_lstm import SpatialStore, build_model, predict_cells, evaluate_scores

ROOT = find_root()
saved_run = Path(globals().get("RUN_DIR", ROOT / "data/interim/conv_lstm/20261027T155222671757Z"))
evaluation_source_spec = json.loads((saved_run / "run.json").read_text())
development_result = json.loads((saved_run / "development_result.json").read_text())
LENGTH = evaluation_source_spec["sequence_length"]
BATCH_SIZE = evaluation_source_spec["batch_size"]
POSITIVE_WEIGHT = evaluation_source_spec["positive_weight"]
LEARNING_RATE = evaluation_source_spec["learning_rate"]
selected_epochs = development_result["selected_epochs"]

# Keep the source export, predictor order and geographic addresses unchanged.
export_info = evaluation_source_spec["source_build"]
source_build = ROOT / export_info["build_directory"]
source_manifest = json.loads((source_build / "manifest.json").read_text())
assert sha256(ROOT / "data/interim/model_table/train_2020_2024.parquet") == export_info["partitions"]["train"]["sha256"]
assert source_manifest["predictors"] == PREDICTORS
study = load_study(root=ROOT, profile="legacy", as_of="2026-10-25")
evaluation_cells, _, _, _ = _grid(study)
for relative in ["data/interim/study_grid_5km/study_grid_5km.csv",
                 "data/interim/study_grid_5km/manifest.json"]:
    assert sha256(ROOT / relative) == source_manifest["descriptor"]["sources"][relative]

scaling_dir = saved_run / "development_scaling"
scaling_record = json.loads((scaling_dir / "scaling.json").read_text())
assert scaling_record["fit_start"] == "2020-01-01"
assert scaling_record["fit_end"] == "2023-09-30"
assert scaling_record["predictors"] == PREDICTORS
# prepare_store sorted the cell axis by cell_id when writing these arrays.
evaluation_store = SpatialStore(
    dates=pd.date_range("2020-01-01", "2024-12-31"),
    cells=evaluation_cells.sort_values("cell_id").reset_index(drop=True),
    features=np.load(scaling_dir / "features.npy", mmap_mode="r"),
    observed=np.load(scaling_dir / "observed.npy", mmap_mode="r"),
    labels=np.load(scaling_dir / "labels.npy", mmap_mode="r"),
    scaling=scaling_record["fitted_ranges"],
)
assert evaluation_store.features.shape == (len(evaluation_store.dates), len(evaluation_store.cells), len(PREDICTORS))
assert evaluation_store.observed.shape == evaluation_store.labels.shape == evaluation_store.features.shape[:2]
fit_targets = evaluation_store.targets("2020-01-01", "2023-09-30", LENGTH)
validation_targets = evaluation_store.targets("2023-10-01", "2023-12-31", LENGTH)
test_targets = evaluation_store.targets("2024-01-01", "2024-12-31", LENGTH)
assert evaluation_store.dates[fit_targets].max() < evaluation_store.dates[validation_targets].min()
assert evaluation_store.dates[validation_targets].max() < evaluation_store.dates[test_targets].min()

checkpoint = saved_run / "development_best.weights.h5"
model = build_model(LENGTH, len(PREDICTORS)+2, POSITIVE_WEIGHT, LEARNING_RATE,
                    spatial_shape=(evaluation_store.height, evaluation_store.width))
model.optimizer.build(model.trainable_variables)
model.load_weights(checkpoint)
model.trainable = False  # Inference only; batch-normalization statistics stay fixed.
EVALUATION_DIR = saved_run / "development_2024_evaluation"
EVALUATION_DIR.mkdir(exist_ok=True)
evaluation_spec = {
    "experiment": "validation-selected development model; no full-year refit",
    "fit_interval": ["2020-01-01", "2023-09-30"],
    "validation_interval": ["2023-10-01", "2023-12-31"],
    "evaluation_interval": ["2024-01-01", "2024-12-31"],
    "selected_epochs": selected_epochs, "stopping_epoch": development_result["last_completed_epoch"],
    "checkpoint": str(checkpoint.relative_to(ROOT)), "checkpoint_sha256": sha256(checkpoint),
    "scaling_file": str((scaling_dir / "scaling.json").relative_to(ROOT)),
    "scaling_sha256": sha256(scaling_dir / "scaling.json"),
    "source_build": export_info, "sequence_length": LENGTH, "horizon_days": 0,
    "historical_fire_input": False, "full_year_refit": False,
    "tensorflow": tf.__version__, "keras": tf.keras.__version__,
    "status": "checkpoint loaded; raw evaluation pending",
}
evaluation_record = EVALUATION_DIR / "evaluation_spec.json"
if evaluation_record.exists():
    recorded = json.loads(evaluation_record.read_text())
    if (recorded.get("checkpoint_sha256") == evaluation_spec["checkpoint_sha256"]
            and recorded.get("scaling_sha256") == evaluation_spec["scaling_sha256"]
            and recorded.get("status") == "raw 2024 evaluation complete; no calibration fitted"):
        evaluation_spec = recorded  # Keep the completed result when simply reopening the model.
evaluation_record.write_text(json.dumps(evaluation_spec, indent=2) + "\n")
print(f"Loaded epoch {selected_epochs}; trained through September 2023, validated October–December 2023.")
print("No fit performed. Evaluation artifacts:", EVALUATION_DIR.relative_to(ROOT))


Loaded epoch 23; trained through September 2023, validated October–December 2023.
No fit performed. Evaluation artifacts: data/interim/conv_lstm/20261027T155222671757Z/development_2024_evaluation


## 6. Evaluate the saved model on 2024 — no fitting

**Training: Jan 2020–Sep 2023. Validation: Oct–Dec 2023. Evaluation: all of 2024.** The saved best checkpoint and its September-fitted scaler are fixed before evaluation. The constant baseline uses the fitting-period frequency only, excluding validation labels. This differs from the paper and section 4 of the classical baseline, whose training uses all of 2020–2023.

Predict only valid study cell-days and save explicit `(cell_id, acq_date)` keys. Report ROC–AUC, average precision, Brier against a constant estimated from the valid training targets, and detection/false-alarm counts. The headline **0.845 is ROC–AUC, not accuracy**. The paper reports raw Brier ≈0.198 and calibrated Brier ≈0.011 (§3.1, pp. 23–24); these are reference values, not expected reproduced results.

The 0.5 rule is fixed. Targets of 100, 1,000 and 10,000 alerts describe the highest-ranked cell-days across the whole year and include every tie. They are not operational daily budgets or thresholds independently selected for deployment. “False alarm” means no qualifying recorded detection, not confirmed absence of fire.

In [7]:
# Calculate reference frequency from actual fitting targets, not validation labels.
training_positive_count = sum(int((evaluation_store.labels[i][evaluation_store.valid_cells(i, LENGTH)] == 1).sum()) for i in fit_targets)
training_valid_count = sum(int(evaluation_store.valid_cells(i, LENGTH).sum()) for i in fit_targets)
training_rate = training_positive_count / training_valid_count
coverage = []
for role, targets, start, end in [
    ("Training", fit_targets, "2020-01-01", "2023-09-30"),
    ("Validation", validation_targets, "2023-10-01", "2023-12-31"),
    ("Evaluation", test_targets, "2024-01-01", "2024-12-31"),
]:
    valid = sum(int(evaluation_store.valid_cells(i, LENGTH).sum()) for i in targets)
    positives = sum(int((evaluation_store.labels[i][evaluation_store.valid_cells(i, LENGTH)] == 1).sum()) for i in targets)
    requested = len(pd.date_range(start, end))*len(evaluation_store.cells)
    coverage.append({"Role": role, "Requested cell-days": requested, "Scored cell-days": valid,
                     "Excluded cell-days": requested-valid, "Positive cell-days": positives})
display(pd.DataFrame(coverage))
print(f"Predicting {len(test_targets)} daily maps for 2024; no training updates.", flush=True)
optimizer_steps_before = int(model.optimizer.iterations.numpy())
predictions = predict_cells(model, evaluation_store, test_targets, LENGTH, BATCH_SIZE)
assert int(model.optimizer.iterations.numpy()) == optimizer_steps_before
assert sha256(checkpoint) == evaluation_spec["checkpoint_sha256"]
assert predictions.acq_date.between("2024-01-01", "2024-12-31").all()
assert len(predictions) == coverage[-1]["Scored cell-days"]
predictions.to_parquet(EVALUATION_DIR / "raw_2024_predictions.parquet", index=False)
raw_metrics, raw_alerts = evaluate_scores(predictions.fire_detected, predictions.raw_probability, training_rate)
display(pd.DataFrame([raw_metrics]).round(6))
display(raw_alerts.style.format({"Threshold": "{:.6g}"}, thousands=","))
(EVALUATION_DIR / "raw_metrics.json").write_text(json.dumps(raw_metrics, indent=2) + "\n")
raw_alerts.to_csv(EVALUATION_DIR / "raw_alert_counts.csv", index=False)
pd.DataFrame(coverage).to_csv(EVALUATION_DIR / "coverage.csv", index=False)
evaluation_spec.update(status="raw 2024 evaluation complete; no calibration fitted",
                       training_constant_rate=training_rate)
(EVALUATION_DIR / "evaluation_spec.json").write_text(json.dumps(evaluation_spec, indent=2) + "\n")
development_result.update(evaluation_2024_complete=True, full_year_refit=False,
                          evaluation_directory="development_2024_evaluation")
(saved_run / "development_result.json").write_text(json.dumps(development_result, indent=2) + "\n")
evaluation_source_spec.update(status="saved development model evaluated on 2024; no full-year refit",
                              full_year_refit=False, selected_epochs=selected_epochs,
                              evaluation_directory="development_2024_evaluation")
(saved_run / "run.json").write_text(json.dumps(evaluation_source_spec, indent=2) + "\n")


,Role,Requested cell-days,Scored cell-days,Excluded cell-days,Positive cell-days
0,Training,4807928,4786856,21072,607
1,Validation,323104,323104,0,332
2,Evaluation,1285392,1285392,0,70


Predicting 366 daily maps for 2024; no training updates.


,Cell-days,Positives,ROC–AUC,Average precision,Prevalence,AP / chance,Brier,Constant Brier,Brier (baseline=100)
0,1285392,70,0.878043,0.002238,0.000054,41.100469,0.008546,0.000054,15692.750682


,Rule,Threshold,Actual alerts,Found,False alarms,Missed,True negatives
0,Fixed 0.5,0.5,"9,228",27,"9,201",43,"1,276,121"
1,Target 100 alerts,0.974271,100,0,100,70,"1,285,222"
2,"Target 1,000 alerts",0.866166,"1,000",4,996,66,"1,284,326"
3,"Target 10,000 alerts",0.486724,"10,000",28,"9,972",42,"1,275,350"


1571

## 7. Isotonic calibration: separate fitting and evaluation

The paper does not state which observations fitted isotonic regression or whether its Brier result used separate observations. The calendar-halves experiment below is an explicit extension: fit a **new ConvLSTM calibrator** on January–June 2024, compare on July–December only. The baseline notebook found just nine positives in the fitting half; the exact count after sequence masking is printed here. These already-viewed observations provide exploratory evidence, not a pristine test.

Raw full-year results above remain the main architecture result. Compare both versions and both frequency constants on the same later cohort. Isotonic regression can merge scores into ties and increase alert counts; more detections with more alerts is not necessarily an improvement.

Calibration is enabled by default (`RUN_CALIBRATION = True`); set it to False in section 1.1 to skip. This cell also runs independently in a fresh kernel using the configured saved run: it reads section 6's saved predictions and experiment record without loading TensorFlow, rebuilding sequences, or running inference. For another run, set `RUN_DIR` explicitly. It saves the fitted mapping, keyed later-cohort predictions, metrics, alert counts and a fingerprinted calibration record. The raw evaluation artifacts retain their original provenance.


In [1]:
# This cell can run on its own after section 6 has saved raw predictions.
if globals().get("RUN_CALIBRATION", True):
    import json
    from datetime import datetime, timezone
    from pathlib import Path

    import joblib
    import numpy as np
    import pandas as pd
    import sklearn
    from IPython.display import display
    from sklearn.isotonic import IsotonicRegression
    from peatguard.conv_lstm import evaluate_scores
    from peatguard.inventory import sha256
    from peatguard.study import find_root

    ROOT = find_root()
    calibration_run = Path(globals().get("RUN_DIR", ROOT / "data/interim/conv_lstm/20261027T155222671757Z"))
    EVALUATION_DIR = calibration_run / "development_2024_evaluation"
    prediction_file = EVALUATION_DIR / "raw_2024_predictions.parquet"
    raw_record_file = EVALUATION_DIR / "evaluation_spec.json"
    raw_record = json.loads(raw_record_file.read_text())
    if raw_record.get("status") != "raw 2024 evaluation complete; no calibration fitted":
        raise ValueError("Complete section 6 raw evaluation before fitting calibration.")
    for path_key, hash_key in [("checkpoint", "checkpoint_sha256"), ("scaling_file", "scaling_sha256")]:
        if sha256(ROOT / raw_record[path_key]) != raw_record[hash_key]:
            raise ValueError(f"Saved {path_key} differs from the raw evaluation record.")
    if pd.Timestamp(raw_record["validation_interval"][1]) >= pd.Timestamp("2024-01-01"):
        raise ValueError("Model selection must end before the calibration cohort.")
    if pd.Timestamp(raw_record["fit_interval"][1]) >= pd.Timestamp("2024-01-01"):
        raise ValueError("Model fitting must end before the calibration cohort.")

    predictions = pd.read_parquet(prediction_file)
    dates = pd.to_datetime(predictions.acq_date, errors="raise")
    if (dates.isna().any() or not dates.between("2024-01-01", "2024-12-31").all()
            or not dates.eq(dates.dt.normalize()).all()):
        raise ValueError("Expected daily 2024 prediction dates only.")
    predictions = predictions.assign(acq_date=dates)
    if predictions.cell_id.isna().any() or predictions.duplicated(["cell_id", "acq_date"]).any():
        raise ValueError("Expected unique, nonmissing cell/date keys.")
    scores = predictions.raw_probability.to_numpy(dtype=float)
    if (not np.isin(predictions.fire_detected, [0, 1]).all() or not np.isfinite(scores).all()
            or not ((scores >= 0) & (scores <= 1)).all()):
        raise ValueError("Expected binary labels and finite probabilities in [0, 1].")
    training_rate = float(raw_record["training_constant_rate"])
    if not np.isfinite(training_rate) or not 0 <= training_rate <= 1:
        raise ValueError("Invalid training-frequency constant.")
    fit_rows = predictions.loc[dates < "2024-07-01"]
    eval_rows = predictions.loc[dates >= "2024-07-01"].copy()
    if fit_rows.fire_detected.nunique() != 2 or eval_rows.fire_detected.nunique() != 2:
        raise ValueError("Both calendar halves require positive and negative observations.")
    print(f"Calibration: {len(fit_rows):,} cell-days; {int(fit_rows.fire_detected.sum())} positives. "
          f"Evaluation: {len(eval_rows):,} cell-days; {int(eval_rows.fire_detected.sum())} positives.")
    calibrator = IsotonicRegression(out_of_bounds="clip")
    calibrator.fit(fit_rows.raw_probability, fit_rows.fire_detected)
    calibration_rate = float(fit_rows.fire_detected.mean())
    versions = {
        "Raw": eval_rows.raw_probability.to_numpy(),
        "Isotonic": calibrator.predict(eval_rows.raw_probability),
        "Training-frequency constant": np.full(len(eval_rows), training_rate),
        "Calibration-frequency constant": np.full(len(eval_rows), calibration_rate),
    }
    metric_rows, alert_frames = [], []
    for name, probabilities in versions.items():
        metrics, alerts = evaluate_scores(eval_rows.fire_detected, probabilities, calibration_rate)
        metric_rows.append({"Version": name, **metrics})
        alert_frames.append(alerts.assign(Version=name))
    calibration_metrics = pd.DataFrame(metric_rows)
    calibration_alerts = pd.concat(alert_frames, ignore_index=True)
    display(calibration_metrics.style.format(precision=9, thousands=","))
    display(calibration_alerts.style.format({"Threshold": "{:.6g}"}, thousands=","))
    joblib.dump(calibrator, EVALUATION_DIR / "isotonic_jan_jun_2024.joblib")
    calibration_metrics.to_csv(EVALUATION_DIR / "calibration_metrics.csv", index=False)
    calibration_alerts.to_csv(EVALUATION_DIR / "calibration_alert_counts.csv", index=False)
    eval_rows.assign(calibrated_probability=versions["Isotonic"]).to_parquet(
        EVALUATION_DIR / "calibration_evaluation_jul_dec_2024.parquet", index=False,
    )
    calibration_spec = {
        "status": "isotonic calibration and separate later-cohort evaluation complete",
        "experiment": "exploratory calendar-halves calibration; previously viewed 2024 outcomes",
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        "method": "sklearn.isotonic.IsotonicRegression", "out_of_bounds": "clip",
        "sklearn": sklearn.__version__, "joblib": joblib.__version__,
        "calibration_fit_interval": ["2024-01-01", "2024-06-30"],
        "calibration_evaluation_interval": ["2024-07-01", "2024-12-31"],
        "calibration_fit_cell_days": len(fit_rows),
        "calibration_fit_positives": int(fit_rows.fire_detected.sum()),
        "evaluation_cell_days": len(eval_rows), "evaluation_positives": int(eval_rows.fire_detected.sum()),
        "training_constant_rate": training_rate, "calibration_constant_rate": calibration_rate,
        "raw_predictions_sha256": sha256(prediction_file),
        "raw_evaluation_spec_sha256": sha256(raw_record_file),
        "checkpoint_sha256": raw_record["checkpoint_sha256"],
        "scaling_sha256": raw_record["scaling_sha256"],
        "artifacts": {name: sha256(EVALUATION_DIR / name) for name in [
            "isotonic_jan_jun_2024.joblib", "calibration_metrics.csv", "calibration_alert_counts.csv",
            "calibration_evaluation_jul_dec_2024.parquet",
        ]},
    }
    (EVALUATION_DIR / "calibration_spec.json").write_text(json.dumps(calibration_spec, indent=2) + "\n")
else:
    print("Calibration skipped (RUN_CALIBRATION = False).")


Calibration: 639,184 cell-days; 9 positives. Evaluation: 646,208 cell-days; 61 positives.


,Version,Cell-days,Positives,ROC–AUC,Average precision,Prevalence,AP / chance,Brier,Constant Brier,Brier (baseline=100)
0,Raw,"646,208",61,0.860607495,0.002491982,0.000094397,26.398998256,0.012477830,0.000094394,"13,218.825806509"
1,Isotonic,"646,208",61,0.841654111,0.002068616,0.000094397,21.914036570,0.000094272,0.000094394,99.870664904
2,Training-frequency constant,"646,208",61,0.500000000,0.000094397,0.000094397,1.000000000,0.000094389,0.000094394,99.994278899
3,Calibration-frequency constant,"646,208",61,0.500000000,0.000094397,0.000094397,1.000000000,0.000094394,0.000094394,100.000000000


,Rule,Threshold,Actual alerts,Found,False alarms,Missed,True negatives,Version
0,Fixed 0.5,0.5,"7,158",22,"7,136",39,"639,011",Raw
1,Target 100 alerts,0.974033,100,0,100,61,"646,047",Raw
2,"Target 1,000 alerts",0.848603,"1,000",4,996,57,"645,151",Raw
3,"Target 10,000 alerts",0.44557,"10,000",27,"9,973",34,"636,174",Raw
4,Fixed 0.5,0.5,0,0,0,61,"646,147",Isotonic
5,Target 100 alerts,0.00278552,718,2,716,59,"645,431",Isotonic
6,"Target 1,000 alerts",0.00278552,"4,201",17,"4,184",44,"641,963",Isotonic
7,"Target 10,000 alerts",0.000441501,"14,850",32,"14,818",29,"631,329",Isotonic
8,Fixed 0.5,0.5,0,0,0,61,"646,147",Training-frequency constant
9,Target 100 alerts,0.000126806,"646,208",61,"646,147",0,0,Training-frequency constant


## 8. Interpretation and limits

### Recorded raw evaluation — 2026-10-28

**Training: Jan 2020–Sep 2023 (607 positive cell-days). Validation: Oct–Dec 2023 (332 positives). Evaluation: 2024 (70 positives).** The validation-selected epoch-23 model was evaluated without a full-year refit. All 1,285,392 evaluation cell-days had usable sequence history. These full-year metrics use raw probabilities; the separate calibration experiment is reported below.

| Metric | Saved-model result |
| --- | ---: |
| ROC–AUC | 0.8780 |
| Average precision | 0.002238 (41.1 × evaluation prevalence) |
| Raw Brier score | 0.008546 |
| Fitting-frequency constant Brier score | 0.00005446 |

| Descriptive rule | Actual alerts | Found positives | False alarms | Missed positives |
| --- | ---: | ---: | ---: | ---: |
| Fixed score ≥ 0.5 | 9,228 | 27 | 9,201 | 43 |
| Highest 1,000 scores, including ties | 1,000 | 4 | 996 | 66 |
| Highest 10,000 scores, including ties | 10,000 | 28 | 9,972 | 42 |

The model has ranking signal, but its raw Brier error is approximately **157 times** the fitting-frequency constant's error, and false alarms dominate the flagged cell-days. These results do not establish reliable probability estimates or operational alerts. ROC–AUC 0.878 must not be interpreted as 87.8% accuracy. The training cohort differs from both the paper and the full-year classical baseline; there is no controlled architecture-only superiority claim. A false alarm denotes no qualifying recorded satellite detection, not independently verified absence of fire.

The checkpoint, September-fitted scaler, raw predictions, coverage, metrics and explicit experiment record are retained under `data/interim/conv_lstm/20261027T155222671757Z/development_2024_evaluation/`. Recompute this interpretation if inputs or the fixed checkpoint change. Do not use 2024 outcomes to tune architecture or select a replacement checkpoint.

### Recorded isotonic calibration — 2026-10-28

A new mapping was fitted on **639,184 January–June 2024 cell-days (9 positives)** and evaluated on **646,208 July–December cell-days (61 positives)**. No network training or checkpoint selection was performed. Compare the following scores only within this later cohort; they have different scoring support from the full-year table above.

| Version | July–December Brier | ROC–AUC | Average precision |
| --- | ---: | ---: | ---: |
| Raw | 0.0124778301 | 0.860607 | 0.002492 |
| Isotonic | 0.0000942723 | 0.841654 | 0.002069 |
| Training-frequency constant | 0.0000943890 | 0.500000 | 0.0000944 |
| Calibration-frequency constant | 0.0000943944 | 0.500000 | 0.0000944 |

Isotonic greatly reduces raw probability error, but beats the calibration-frequency constant by only **0.129%** and reduces both ranking metrics through ties. At score ≥0.5 it produces no alerts. The descriptive 10,000-alert target expands to **14,850 alerts**, finding 32 positives with 14,818 false alarms; the raw version found 27 with 10,000 alerts. These unequal volumes do not establish improved detection at an equal alert budget. Nine calibration positives and seasonal differences limit confidence in the mapping. This is exploratory calibration on previously viewed observations, not an untouched test or evidence of operational readiness.

The fitted mapping and keyed later-cohort predictions remain in the local evaluation directory. Small metric/count tables and `calibration_spec.json` are copied to `results/conv_lstm/validation_2023q4_evaluation_2024/` so the recorded outcome can be reviewed without local model artifacts.

- The model was trained Jan 2020–Sep 2023 and selected using Oct–Dec 2023 validation, then evaluated on 2024 with no full-year refit. Section 4 of `base_models.ipynb` used all of 2020–2023 for fitting: its training cohort differs, so this is not a controlled same-training-cohort comparison. State the exact scored cohort and any sequence exclusions. Spatial and temporal context differ from the tabular inputs, so performance differences cannot be attributed solely to architecture.
- Assess ranking and probability error separately. Compare Brier with its constant baseline and inspect detection counts before describing useful alerts. Class weighting alone does not yield calibrated probabilities.
- Compare calibration versions only within July–December; there are few calibration positives and a seasonal distribution shift. The paper's seasonal Brier scores (0.088 dry, 0.027 wet) cannot average to its overall calibrated 0.011 on the same cohort; the calibration/scoring support is unresolved.
- Sequence length, same-day timing, no historical-fire inputs, mask channels, padding, optimizer and epoch-selection procedure are reconstruction assumptions. Annual land cover and retrospective weather do not establish operational availability. The candidate BIG mask remains pending scientific review.
- This notebook implements a reproducible methodological reconstruction. It does not claim exact numerical reproduction, independent validation of the published weight search, or three-/ten-day forecasting capability.